In [1]:
"""
FSDP Compatibility Test for OpenPi0ForRLActionPrediction

This notebook tests if the model is compatible with PyTorch FSDP wrapping.
"""
import os
import sys

# Set environment for single-GPU testing
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ["LOCAL_RANK"] = "0"
os.environ["RANK"] = "0"
os.environ["WORLD_SIZE"] = "1"
os.environ["MASTER_ADDR"] = "localhost"
os.environ["MASTER_PORT"] = "29500"

import torch
import torch.distributed as dist
from torch.distributed.fsdp import FullyShardedDataParallel as FSDP
from torch.distributed.fsdp.wrap import (
    transformer_auto_wrap_policy,
    size_based_auto_wrap_policy,
)
import functools

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA device: {torch.cuda.get_device_name(0)}")

PyTorch version: 2.6.0+cu124
CUDA available: True
CUDA device: NVIDIA H100 80GB HBM3


In [2]:
# Initialize distributed (required for FSDP)
if not dist.is_initialized():
    dist.init_process_group(backend="nccl", init_method="env://")
    print(f"Distributed initialized: rank={dist.get_rank()}, world_size={dist.get_world_size()}")


Distributed initialized: rank=0, world_size=1


In [3]:
# Load the OpenPi0 model
from omegaconf import OmegaConf
import sys
sys.path.append("/home/ubuntu/haitong-south-2/RLinf")
from rlinf.models.embodiment.openpi import get_model

# Configure model - update model_path to your actual model path
model_cfg = OmegaConf.create({
    "model_path": "/home/ubuntu/cache/models/RLinf-Pi0-LIBERO-Spatial-Object-Goal-SFT",  # UPDATE THIS
    "openpi": {
        "config_name": "pi0_libero",
        "num_images_in_input": 2,
        "action_chunk": 5,
        "action_env_dim": 7,
        "num_steps": 10,
        "add_value_head": False,
    }
})

print("Loading OpenPi0 model...")
model = get_model(model_cfg)
model = model.cuda()
print(f"Model loaded: {type(model).__name__}")


/home/ubuntu/haitong-south-2/RLinf/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2026-01-26 15:36:21,208	INFO util.py:154 -- Missing packages: ['ipywidgets']. Run `pip install -U ipywidgets`, then restart the notebook server for rich notebook output.


Loading OpenPi0 model...


INFO:datasets:PyTorch version 2.6.0 available.
INFO:datasets:Polars version 1.37.1 available.
INFO:datasets:TensorFlow version 2.20.0 available.
INFO:datasets:JAX version 0.5.3 available.
INFO:root:Loaded norm stats from /home/ubuntu/cache/models/RLinf-Pi0-LIBERO-Spatial-Object-Goal-SFT/physical-intelligence/libero
INFO:root:Loaded norm stats from /home/ubuntu/cache/models/RLinf-Pi0-LIBERO-Spatial-Object-Goal-SFT/physical-intelligence/libero


Model loaded: OpenPi0ForRLActionPrediction


In [10]:
# Check model's FSDP-related attributes
print("=== FSDP Compatibility Check ===\n")

# Check _no_split_modules
no_split_modules = getattr(model, "_no_split_modules", None)
print(f"_no_split_modules: {no_split_modules}")

# Check _no_split_names (custom attribute in OpenPi0)
no_split_names = getattr(model, "_no_split_names", None)
print(f"_no_split_names: {no_split_names}")

# List all unique module class names
module_classes = set()
for name, module in model.named_modules():
    module_classes.add(module.__class__.__name__)
print(f"\nUnique module classes ({len(module_classes)}):")
for cls in sorted(module_classes):
    print(f"  - {cls}")


=== FSDP Compatibility Check ===

_no_split_modules: ['GemmaMLP', 'SiglipVisionEmbeddings', 'GemmaRMSNorm', 'GemmaRotaryEmbedding']
_no_split_names: ['action_in_proj', 'action_out_proj', 'lm_head', 'state_proj', 'action_time_mlp_in', 'action_time_mlp_out', 'time_mlp_in', 'time_mlp_out']

Unique module classes (25):
  - Conv2d
  - Embedding
  - GemmaAttention
  - GemmaDecoderLayer
  - GemmaForCausalLM
  - GemmaMLP
  - GemmaModel
  - GemmaRMSNorm
  - GemmaRotaryEmbedding
  - LayerNorm
  - Linear
  - ModuleList
  - OpenPi0ForRLActionPrediction
  - PaliGemmaForConditionalGeneration
  - PaliGemmaModel
  - PaliGemmaMultiModalProjector
  - PaliGemmaWithExpertModel
  - PytorchGELUTanh
  - SiglipAttention
  - SiglipEncoder
  - SiglipEncoderLayer
  - SiglipMLP
  - SiglipVisionEmbeddings
  - SiglipVisionModel
  - SiglipVisionTransformer


In [5]:
model.action_out_proj

Linear(in_features=1024, out_features=32, bias=True)

In [17]:
# Check parameter dtypes - the model has mixed dtypes which causes FSDP1 issues
print("=== Parameter dtype distribution ===")
dtype_counts = {}
for name, param in model.named_parameters():
    dtype = str(param.dtype)
    dtype_counts[dtype] = dtype_counts.get(dtype, 0) + 1

for dtype, count in sorted(dtype_counts.items()):
    print(f"  {dtype}: {count} parameters")

# Fix: Convert all parameters to bfloat16 for uniform dtype
print("\nConverting all parameters to bfloat16...")
model = model.to(dtype=torch.bfloat16)

# Verify conversion
dtype_counts_after = {}
for name, param in model.named_parameters():
    dtype = str(param.dtype)
    dtype_counts_after[dtype] = dtype_counts_after.get(dtype, 0) + 1
print("After conversion:")
for dtype, count in sorted(dtype_counts_after.items()):
    print(f"  {dtype}: {count} parameters")


=== Parameter dtype distribution ===
  torch.bfloat16: 777 parameters

Converting all parameters to bfloat16...
After conversion:
  torch.bfloat16: 777 parameters


In [18]:
# Setup FSDP wrap policy
from torch.distributed.fsdp import MixedPrecision

# Get the wrap policy classes
wrap_classes = set()
for cls_name in no_split_modules or []:
    for name, module in model.named_modules():
        if module.__class__.__name__ == cls_name:
            wrap_classes.add(type(module))
            break

print(f"Classes to wrap with FSDP: {wrap_classes}")

# Create auto wrap policy
if wrap_classes:
    auto_wrap_policy = functools.partial(
        transformer_auto_wrap_policy,
        transformer_layer_cls=wrap_classes,
    )
else:
    auto_wrap_policy = functools.partial(
        size_based_auto_wrap_policy,
        min_num_params=1e6,
    )

# Mixed precision config (keeping bfloat16 since model is now uniform)
mp_policy = MixedPrecision(
    param_dtype=torch.bfloat16,
    reduce_dtype=torch.bfloat16,
    buffer_dtype=torch.bfloat16,
)

# Wrap model with FSDP
print("\nWrapping model with FSDP...")
try:
    fsdp_model = FSDP(
        model,
        auto_wrap_policy=auto_wrap_policy,
        mixed_precision=mp_policy,
        device_id=torch.cuda.current_device(),
        use_orig_params=True,
    )
    print("✓ FSDP wrapping successful!")
    print(f"FSDP model type: {type(fsdp_model)}")
except Exception as e:
    print(f"✗ FSDP wrapping failed: {e}")
    import traceback
    traceback.print_exc()


Classes to wrap with FSDP: {<class 'transformers.models.gemma.modeling_gemma.GemmaMLP'>, <class 'transformers.models.gemma.modeling_gemma.GemmaRMSNorm'>, <class 'transformers.models.gemma.modeling_gemma.GemmaRotaryEmbedding'>, <class 'transformers.models.siglip.modeling_siglip.SiglipVisionEmbeddings'>}

Wrapping model with FSDP...
✗ FSDP wrapping failed: FSDP auto wrapping requires modules to not already have FSDP applied but found paligemma_with_expert.paligemma.model.vision_tower.vision_model.embeddings in
OpenPi0ForRLActionPrediction(
  (paligemma_with_expert): PaliGemmaWithExpertModel(
    (paligemma): PaliGemmaForConditionalGeneration(
      (model): PaliGemmaModel(
        (vision_tower): SiglipVisionModel(
          (vision_model): SiglipVisionTransformer(
            (embeddings): FullyShardedDataParallel(
              (_fsdp_wrapped_module): SiglipVisionEmbeddings(
                (patch_embedding): Conv2d(3, 1152, kernel_size=(14, 14), stride=(14, 14), padding=valid)
       

Traceback (most recent call last):
  File "/tmp/ipykernel_722726/3252152813.py", line 36, in <module>
    fsdp_model = FSDP(
                 ^^^^^
  File "/home/ubuntu/haitong-south-2/RLinf/.venv/lib/python3.11/site-packages/torch/distributed/fsdp/fully_sharded_data_parallel.py", line 483, in __init__
    _auto_wrap(
  File "/home/ubuntu/haitong-south-2/RLinf/.venv/lib/python3.11/site-packages/torch/distributed/fsdp/_wrap_utils.py", line 45, in _auto_wrap
    _check_nested_wrapping(root_module)
  File "/home/ubuntu/haitong-south-2/RLinf/.venv/lib/python3.11/site-packages/torch/distributed/fsdp/_wrap_utils.py", line 107, in _check_nested_wrapping
    raise ValueError(
ValueError: FSDP auto wrapping requires modules to not already have FSDP applied but found paligemma_with_expert.paligemma.model.vision_tower.vision_model.embeddings in
OpenPi0ForRLActionPrediction(
  (paligemma_with_expert): PaliGemmaWithExpertModel(
    (paligemma): PaliGemmaForConditionalGeneration(
      (model): Pali

In [22]:
# Test forward pass with dummy data (SFT forward)
import numpy as np
from openpi.models import model as _model

print("\n=== Testing SFT Forward Pass ===")

batch_size = 2
action_horizon = 50
action_dim = 24

# Create dummy observation
dummy_obs = _model.Observation(
    images=[torch.randn(batch_size, 3,  224, 224).cuda()],  # dummy image
    image_masks=[torch.ones(batch_size, dtype=torch.bool).cuda()],
    tokenized_prompt=torch.randint(0, 1000, (batch_size, 48)).cuda(),
    tokenized_prompt_mask=torch.ones(batch_size, 48, dtype=torch.bool).cuda(),
    state=torch.randn(batch_size, 8).cuda(),
)

# Create dummy actions
dummy_actions = torch.randn(batch_size, action_horizon, action_dim).cuda()

print(f"Observation state shape: {dummy_obs.state.shape}")
print(f"Actions shape: {dummy_actions.shape}")

try:
    with torch.no_grad():
        # Test SFT forward
        loss = fsdp_model.module.sft_forward(
            {"observation": dummy_obs, "actions": dummy_actions}
        )
        print(f"✓ SFT forward pass successful! Loss shape: {loss.shape}")
except Exception as e:
    print(f"✗ SFT forward pass failed: {e}")
    import traceback
    traceback.print_exc()



=== Testing SFT Forward Pass ===


TypeCheckError: Type-check error whilst checking the parameters of openpi.models.model.Observation.
The problem arose whilst typechecking parameter 'images'.
Actual value: [f32[2,3,224,224](torch)]
Expected type: dict[str, Union[Float[Array, '*b h w c'], Float[Tensor, '*b h w c'], Float[ndarray, '*b h w c']]].
----------------------
Called with parameters: {
  'self': Observation(...),
  'images': [f32[2,3,224,224](torch)],
  'image_masks': [bool[2](torch)],
  'state': f32[2,8](torch),
  'tokenized_prompt': i64[2,48](torch),
  'tokenized_prompt_mask': bool[2,48](torch),
  'token_ar_mask': None,
  'token_loss_mask': None
}
Parameter annotations: (self: Any, images: dict[str, Union[Float[Array, '*b h w c'], Float[Tensor, '*b h w c'], Float[ndarray, '*b h w c']]], image_masks: dict[str, Union[Bool[Array, '*b'], Bool[Tensor, '*b'], Bool[ndarray, '*b']]], state: Union[Float[Array, '*b s'], Float[Tensor, '*b s'], Float[ndarray, '*b s']], tokenized_prompt: Union[Int[Array, '*b l'], Int[Tensor

In [ ]:
# Test training (forward + backward)
print("\n=== Testing Training (Forward + Backward) ===")

try:
    fsdp_model.train()
    
    # Forward pass
    loss = fsdp_model.module.sft_forward(
        {"observation": dummy_obs, "actions": dummy_actions}
    )
    scalar_loss = loss.mean()
    print(f"Loss: {scalar_loss.item():.6f}")
    
    # Backward pass
    scalar_loss.backward()
    print("✓ Backward pass successful!")
    
    # Check gradients
    grad_count = 0
    none_grad_count = 0
    for name, param in fsdp_model.named_parameters():
        if param.requires_grad:
            if param.grad is not None:
                grad_count += 1
            else:
                none_grad_count += 1
    print(f"Parameters with gradients: {grad_count}")
    print(f"Parameters without gradients: {none_grad_count}")
    
except Exception as e:
    print(f"✗ Training test failed: {e}")
    import traceback
    traceback.print_exc()


In [ ]:
# Alternative: Test with FSDP2 (fully_shard) - the API used in this codebase
print("\n=== Testing FSDP2 (fully_shard) ===")

# Reload model for FSDP2 test
model2 = get_model(model_cfg)
model2 = model2.to(dtype=torch.bfloat16).cuda()  # Fix: uniform dtype

try:
    from torch.distributed._composable.fsdp import fully_shard, MixedPrecisionPolicy
    from torch.distributed.device_mesh import init_device_mesh
    
    # Create device mesh for single GPU
    device_mesh = init_device_mesh("cuda", (1,))
    
    mp_policy_v2 = MixedPrecisionPolicy(
        param_dtype=torch.bfloat16,
        reduce_dtype=torch.bfloat16,
    )
    
    # Wrap with FSDP2
    fsdp2_model = fully_shard(
        model2,
        mesh=device_mesh,
        mp_policy=mp_policy_v2,
    )
    print("✓ FSDP2 (fully_shard) wrapping successful!")
    
    # Test forward
    with torch.no_grad():
        loss2 = fsdp2_model.sft_forward(
            {"observation": dummy_obs, "actions": dummy_actions}
        )
        print(f"✓ FSDP2 forward pass successful! Loss shape: {loss2.shape}")
        
except Exception as e:
    print(f"✗ FSDP2 test failed: {e}")
    import traceback
    traceback.print_exc()


In [ ]:
# Test with the codebase's FSDP utility function (apply_fsdp2_to_model)
print("\n=== Testing with RLinf's apply_fsdp2_to_model ===")

# Reload model
model3 = get_model(model_cfg)
model3 = model3.to(dtype=torch.bfloat16).cuda()  # Fix: uniform dtype

try:
    from rlinf.hybrid_engines.fsdp.utils import apply_fsdp2_to_model
    from torch.distributed._composable.fsdp import MixedPrecisionPolicy, CPUOffloadPolicy, OffloadPolicy
    from torch.distributed.device_mesh import init_device_mesh
    
    device_mesh = init_device_mesh("cuda", (1,))
    
    mp_policy_v3 = MixedPrecisionPolicy(
        param_dtype=torch.bfloat16,
        reduce_dtype=torch.bfloat16,
        cast_forward_inputs=True,
    )
    
    offload_policy = OffloadPolicy()
    
    # Simulated config
    fsdp_config = {
        "wrap_policy": {
            "transformer_layer_cls_to_wrap": model3._no_split_modules
        }
    }
    
    fsdp3_model = apply_fsdp2_to_model(
        module=model3,
        config=fsdp_config,
        device_mesh=device_mesh,
        mp_policy=mp_policy_v3,
        offload_policy=offload_policy,
        reshard_after_forward=True,
    )
    print("✓ apply_fsdp2_to_model wrapping successful!")
    
    # Test forward
    with torch.no_grad():
        loss3 = fsdp3_model.sft_forward(
            {"observation": dummy_obs, "actions": dummy_actions}
        )
        print(f"✓ Forward pass successful! Loss shape: {loss3.shape}")
        
except Exception as e:
    print(f"✗ apply_fsdp2_to_model test failed: {e}")
    import traceback
    traceback.print_exc()



=== Testing with RLinf's apply_fsdp2_to_model ===


NameError: name 'get_model' is not defined

In [ ]:
# Summary and cleanup
print("\n" + "="*50)
print("FSDP COMPATIBILITY TEST SUMMARY")
print("="*50)
print("""
Key findings for OpenPi0ForRLActionPrediction FSDP compatibility:

1. MIXED DTYPE ISSUE: The model loads with mixed dtypes (bfloat16 + float32)
   - FSDP1 requires uniform dtype - must call model.to(dtype=torch.bfloat16) first
   - This happens because some layers (e.g., normalization) default to float32

2. _no_split_modules: The model defines which modules should NOT be split
   across devices (GemmaMLP, SiglipVisionEmbeddings, GemmaRMSNorm, etc.)

3. FSDP1 (FullyShardedDataParallel): Works after dtype unification

4. FSDP2 (fully_shard): Works with the composable API

5. RLinf's apply_fsdp2_to_model: Uses the model's _no_split_modules

Notes:
- gradient_checkpointing is NOT supported for OpenPi0
- use_orig_params=True is recommended for optimizer compatibility
- ALWAYS convert to uniform dtype before FSDP wrapping:
    model = model.to(dtype=torch.bfloat16)
""")

# Cleanup distributed
if dist.is_initialized():
    dist.destroy_process_group()
    print("\nDistributed process group destroyed.")
